# Signal Existence Diagnostic — Sprint 1

Гейт-проверка ПЕРЕД любыми архитектурными работами. Отвечает на вопрос:
**«Есть ли в данных сигнал, который теоретически возможно выучить?»**

После R-0050+R-0051 collapse-эпизодов: сжигать ещё час обучения на iTransformer
не имеет смысла, пока мы не подтвердили простыми тестами, что задача
вообще обучаема.

## Тесты

| # | Тест | Что отвечает | Хорошо если |
|---|---|---|---|
| **S1.3** | Label audit на реальных данных | Метки корректно вычислены? | computed_lr ≈ stored_lr (diff < 1e-4) |
| **S1.2** | LightGBM baseline | Простая tabular-модель находит сигнал? | val_log_loss < baseline_log_loss значимо; AUC > 0.51 |
| **S1.1** | Permutation test | Сигнал статистически значим? | p-value < 0.05 |

## Что НЕ делаем

- НЕ обучаем iTransformer / ансамбли / SVGD
- НЕ калибруем conformal
- НЕ пишем новых loss

Только диагностика. Если все три теста зелёные — продолжаем спринты 2-5.
Если красные — проблема не в архитектуре, а в данных/задаче, и нужно
менять цель или тянуть дополнительные источники.

## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '--quiet',
         'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'lightgbm>=4.0',
         'tqdm>=4.66', 'matplotlib>=3.7'],
        check=True,
    )
    print('Готово.')

In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, roc_auc_score
from tqdm.auto import tqdm

import lightgbm as lgb

from graduate_work.config import default_config

cfg = default_config()
print(f'tickers={cfg.data.tickers}, horizons={cfg.data.horizons}')

## 1. Загрузка данных + построение признаков

Тот же pipeline, что и `algopack_baseline.ipynb` — нужны те же фичи + targets,
что видит iTransformer.

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')
    else:
        raise FileNotFoundError(f'Не найдено {src_dir}')

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features, order_to_trade_ratio,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns


def _load_supercandle(product: str, ticker: str) -> pd.DataFrame:
    p = cfg.paths.data_raw / 'algopack' / product / f'{ticker}.csv'
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
                            utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['open']   = ts['pr_open'].astype(float)
    out['high']   = ts['pr_high'].astype(float)
    out['low']    = ts['pr_low'].astype(float)
    out['close']  = ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


ts_data, os_data, ob_data = {}, {}, {}
for ticker in cfg.data.tickers:
    ts_data[ticker] = _load_supercandle('tradestats', ticker)
    os_data[ticker] = _load_supercandle('orderstats', ticker)
    ob_data[ticker] = _load_supercandle('obstats', ticker)
    print(f'  {ticker}: {len(ts_data[ticker])} баров')

frames = []
for ticker in cfg.data.tickers:
    if ts_data[ticker].empty:
        continue
    feat = _ts_to_ohlcv(ts_data[ticker], ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(tradestats_features(ts_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    feat = feat.fillna(0.0)
    frames.append(feat)

full = pd.concat(frames).sort_index()
print(f'\nfull: {full.shape}')

In [ ]:
out_parts = []
costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
for ticker, sub in full.groupby('ticker'):
    labels = cost_aware_classification_labels(
        open_price=sub['open'], close_price=sub['close'],
        horizons=cfg.data.horizons,
        entry_cost=costs, exit_cost=costs,
        label_smoothing=0.0,
        direction='long',
    )
    out_parts.append(sub.join(labels, how='left'))
full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

target_cols = [f'target_h{h}' for h in cfg.data.horizons]
lr_cols = lr_columns(cfg.data.horizons)
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols]
print(f'frame: {full_with_targets.shape}')
print(f'features: {len(feature_cols)}, targets: {target_cols}')
for c in target_cols:
    p_up = float((full_with_targets[c].dropna() >= 0.5).mean())
    print(f'  {c}: P(UP)={p_up:.3f}')

## 2. S1.3 — Label Audit на РЕАЛЬНЫХ данных

Берём 20 случайных строк из SBER, ВРУЧНУЮ пересчитываем `lr_h6` по формуле
и сравниваем со stored. Если хоть одна не совпадает — проблема в коде
построения меток.

Unit-тесты `tests/test_label_correctness.py` уже проверили формулу на
синтетике; этот тест проверяет на ЖИВЫХ данных, что pipeline не
перепутал индексы / тикеры / NaN.

In [ ]:
TICKER = cfg.data.tickers[0]
AUDIT_HORIZON = 6
AUDIT_N = 20

ticker_df = (
    full_with_targets[full_with_targets['ticker'] == TICKER]
    .sort_index()
    .dropna(subset=[f'lr_h{AUDIT_HORIZON}'])
)
rng = np.random.default_rng(42)
audit_idx = rng.choice(len(ticker_df) - AUDIT_HORIZON - 2, size=AUDIT_N, replace=False)

discrepancies = 0
max_diff = 0.0
rows = []
for idx_pos in audit_idx:
    row_t = ticker_df.iloc[idx_pos]
    row_t1 = ticker_df.iloc[idx_pos + 1]
    row_th = ticker_df.iloc[idx_pos + AUDIT_HORIZON]
    next_open = float(row_t1['open'])
    future_close = float(row_th['close'])
    entry = next_open * (1.0 + costs)
    exit_net = future_close * (1.0 - costs)
    computed_lr = float(np.log(exit_net / entry))
    computed_target = float(computed_lr > 0)
    stored_lr = float(row_t[f'lr_h{AUDIT_HORIZON}'])
    stored_target = float(row_t[f'target_h{AUDIT_HORIZON}'])
    diff = abs(computed_lr - stored_lr)
    max_diff = max(max_diff, diff)
    if diff > 1e-4:
        discrepancies += 1
    rows.append({
        't': row_t.name,
        'next_open': next_open,
        'future_close': future_close,
        'computed_lr': computed_lr,
        'stored_lr': stored_lr,
        'diff': diff,
        'computed_target': computed_target,
        'stored_target': stored_target,
        'match': diff < 1e-4,
    })

audit_df = pd.DataFrame(rows)
print('=== LABEL AUDIT (10 строк из 20) ===')
print(audit_df.head(10).to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print(f'\nMAX |computed - stored| = {max_diff:.2e}')
print(f'Discrepancies (>1e-4): {discrepancies} / {AUDIT_N}')

S13_PASS = discrepancies == 0
print(f'\nS1.3 VERDICT: {"✓ PASS" if S13_PASS else "✗ FAIL — bug в построении меток"}')

## 3. Подготовка данных для S1.2 и S1.1

Используем **не sliding windows** (как для iTransformer), а **просто фичи
последнего бара** — табулярный формат, на котором работает LightGBM.
Если GBM находит сигнал на этом упрощённом представлении, значит
сигнал есть и в обычных фичах, и проблема не в данных.

In [ ]:
from graduate_work.features.pipeline import chronological_split
from graduate_work.features.scaler import StandardScaler

train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)

# Tabular: одна строка = один бар. Берём только feature_cols (без targets/lr).
# dropna по target — LightGBM не учится на NaN-метках.
def _tabular(df: pd.DataFrame, target_col: str) -> tuple[np.ndarray, np.ndarray]:
    valid = df[target_col].notna()
    X = df.loc[valid, feature_cols].to_numpy(dtype=np.float32)
    y = df.loc[valid, target_col].to_numpy(dtype=np.float32)
    return X, y

print(f'train rows = {len(train_df)}')
print(f'val rows   = {len(val_df)}')
print(f'test rows  = {len(test_df)}')

## 4. S1.2 — LightGBM Baseline

Per-horizon binary классификатор на табулярных фичах. Метрики:

- **AUC** — ранжирующая способность (без калибровки порога). Случайно — 0.5.
- **log_loss** vs **prior baseline** (`always predict P(UP)`). Если
  GBM_log_loss < prior_log_loss значимо, GBM учится.
- **Accuracy** vs majority class baseline.

**Pass-критерий**: AUC > 0.51 на val И test (модель ранжирует лучше
случайного), И log_loss < prior_log_loss минимум на 1‰ — иначе
табулярный сигнал в фичах не различим.

In [ ]:
lgb_results = []
for h, target_col in zip(cfg.data.horizons, target_cols):
    X_tr, y_tr = _tabular(train_df, target_col)
    X_va, y_va = _tabular(val_df, target_col)
    X_te, y_te = _tabular(test_df, target_col)
    p_up = float(y_tr.mean())

    model = lgb.LGBMClassifier(
        n_estimators=200, num_leaves=31,
        learning_rate=0.05, max_depth=-1,
        min_child_samples=200,
        reg_lambda=1.0, reg_alpha=0.1,
        random_state=42,
        verbosity=-1, n_jobs=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(20, verbose=False)],
    )
    p_va = model.predict_proba(X_va)[:, 1]
    p_te = model.predict_proba(X_te)[:, 1]

    # Baseline: всегда предсказываем prior из train.
    baseline_va = np.full_like(y_va, p_up)
    baseline_te = np.full_like(y_te, p_up)

    lgb_results.append({
        'horizon': h,
        'P(UP)_train': p_up,
        'auc_val': roc_auc_score(y_va, p_va) if len(np.unique(y_va)) == 2 else float('nan'),
        'auc_test': roc_auc_score(y_te, p_te) if len(np.unique(y_te)) == 2 else float('nan'),
        'logloss_val': log_loss(y_va, np.clip(p_va, 1e-7, 1-1e-7)),
        'logloss_baseline_val': log_loss(y_va, np.clip(baseline_va, 1e-7, 1-1e-7)),
        'logloss_test': log_loss(y_te, np.clip(p_te, 1e-7, 1-1e-7)),
        'logloss_baseline_test': log_loss(y_te, np.clip(baseline_te, 1e-7, 1-1e-7)),
        'best_iter': model.best_iteration_,
    })

lgb_df = pd.DataFrame(lgb_results)
lgb_df['lift_val'] = lgb_df['logloss_baseline_val'] - lgb_df['logloss_val']
lgb_df['lift_test'] = lgb_df['logloss_baseline_test'] - lgb_df['logloss_test']

print('=== LightGBM baseline (per horizon) ===')
print(lgb_df[[
    'horizon', 'P(UP)_train', 'auc_val', 'auc_test',
    'logloss_val', 'logloss_baseline_val', 'lift_val',
    'logloss_test', 'logloss_baseline_test', 'lift_test',
    'best_iter',
]].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

auc_min = float(lgb_df[['auc_val', 'auc_test']].min().min())
lift_min = float(lgb_df[['lift_val', 'lift_test']].min().min())
S12_PASS = (auc_min > 0.51) and (lift_min > 1e-3)
print(f'\nMin AUC across val/test = {auc_min:.4f} (нужно > 0.51)')
print(f'Min log_loss lift across val/test = {lift_min:.5f} (нужно > 0.001)')
print(f'\nS1.2 VERDICT: {"✓ PASS — LightGBM нашёл сигнал" if S12_PASS else "✗ FAIL — табулярного сигнала нет"}')

## 5. S1.1 — Permutation Test

**Ojala & Garriga, JMLR 2010**. Train на shuffled labels N раз → null
распределение val_log_loss. Сравниваем с реальным.

**p-value** = (1 + count(L_null ≤ L_real)) / (1 + N).

**Pass-критерий**: p < 0.05 хотя бы на одном горизонте — сигнал
статистически значим. Для скорости берём подвыборку `SUBSAMPLE`
строк из train и упрощённый LightGBM (`n_estimators=50`).

In [ ]:
SUBSAMPLE = 10_000  # размер train подвыборки
VAL_SUBSAMPLE = 5_000
N_PERM = 30  # итераций null distribution

def _train_eval_logloss(
    X_tr: np.ndarray, y_tr: np.ndarray,
    X_va: np.ndarray, y_va: np.ndarray,
    seed: int = 0,
) -> float:
    """Train fast LightGBM, return val log_loss. Прозрачно для пермутации."""
    model = lgb.LGBMClassifier(
        n_estimators=80, num_leaves=15,
        learning_rate=0.1, max_depth=6,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=seed,
        verbosity=-1, n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    p = np.clip(model.predict_proba(X_va)[:, 1], 1e-7, 1 - 1e-7)
    return float(log_loss(y_va, p))

perm_results = []
rng = np.random.default_rng(0)
for h, target_col in zip(cfg.data.horizons, target_cols):
    X_tr_full, y_tr_full = _tabular(train_df, target_col)
    X_va_full, y_va_full = _tabular(val_df, target_col)
    # Subsample для скорости
    tr_idx = rng.choice(len(X_tr_full), size=min(SUBSAMPLE, len(X_tr_full)), replace=False)
    va_idx = rng.choice(len(X_va_full), size=min(VAL_SUBSAMPLE, len(X_va_full)), replace=False)
    X_tr = X_tr_full[tr_idx]
    y_tr = y_tr_full[tr_idx]
    X_va = X_va_full[va_idx]
    y_va = y_va_full[va_idx]

    # Real run
    L_real = _train_eval_logloss(X_tr, y_tr, X_va, y_va, seed=0)

    # Null distribution
    null_losses = []
    for k in tqdm(range(N_PERM), desc=f'perm h={h}', leave=False):
        y_shuf = rng.permutation(y_tr)
        L_null = _train_eval_logloss(X_tr, y_shuf, X_va, y_va, seed=k+1)
        null_losses.append(L_null)
    null_losses = np.array(null_losses)

    p_value = (1 + (null_losses <= L_real).sum()) / (1 + N_PERM)
    perm_results.append({
        'horizon': h,
        'L_real': L_real,
        'L_null_mean': float(null_losses.mean()),
        'L_null_std': float(null_losses.std()),
        'L_null_min': float(null_losses.min()),
        'p_value': p_value,
    })

perm_df = pd.DataFrame(perm_results)
print('=== PERMUTATION TEST (LightGBM, n=10k train, 30 shuffles) ===')
print(perm_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

min_p = float(perm_df['p_value'].min())
S11_PASS = min_p < 0.05
print(f'\nMin p-value: {min_p:.4f} (нужно < 0.05)')
print(f'\nS1.1 VERDICT: {"✓ PASS — сигнал статистически значим" if S11_PASS else "✗ FAIL — H₀ независимости фичи-метки не отвергнута"}')

In [ ]:
fig, axes = plt.subplots(1, len(cfg.data.horizons), figsize=(14, 3.5), sharey=True)
if len(cfg.data.horizons) == 1:
    axes = [axes]

rng = np.random.default_rng(0)
for ax, h, target_col in zip(axes, cfg.data.horizons, target_cols):
    X_tr_full, y_tr_full = _tabular(train_df, target_col)
    X_va_full, y_va_full = _tabular(val_df, target_col)
    tr_idx = rng.choice(len(X_tr_full), size=min(SUBSAMPLE, len(X_tr_full)), replace=False)
    va_idx = rng.choice(len(X_va_full), size=min(VAL_SUBSAMPLE, len(X_va_full)), replace=False)
    X_tr = X_tr_full[tr_idx]; y_tr = y_tr_full[tr_idx]
    X_va = X_va_full[va_idx]; y_va = y_va_full[va_idx]
    L_real = _train_eval_logloss(X_tr, y_tr, X_va, y_va, seed=0)
    nulls = []
    for k in range(N_PERM):
        y_shuf = rng.permutation(y_tr)
        nulls.append(_train_eval_logloss(X_tr, y_shuf, X_va, y_va, seed=k+1))
    ax.hist(nulls, bins=20, alpha=0.7, color='steelblue', label='null (shuffled)')
    ax.axvline(L_real, color='red', lw=2, label=f'real={L_real:.4f}')
    ax.set_title(f'h={h}')
    ax.set_xlabel('val log_loss')
    ax.legend()
axes[0].set_ylabel('count')
fig.suptitle('Permutation null distribution: real vs shuffled-label model')
plt.tight_layout(); plt.show()

## 6. ИТОГОВЫЙ VERDICT

In [ ]:
verdict = pd.DataFrame([
    {'sprint': 'S1.3 Label audit',     'pass': S13_PASS, 'meaning': 'Метки построены корректно' if S13_PASS else 'Bug в построении меток!'},
    {'sprint': 'S1.2 LightGBM',         'pass': S12_PASS, 'meaning': 'Табулярный сигнал есть' if S12_PASS else 'GBM не находит сигнал'},
    {'sprint': 'S1.1 Permutation test', 'pass': S11_PASS, 'meaning': 'Сигнал статистически значим' if S11_PASS else 'H₀ не отвергнута'},
])
print('=== SPRINT 1 VERDICT ===')
print(verdict.to_string(index=False))
all_pass = bool(verdict['pass'].all())
print(f'\nИтог: {"✓ ALL GREEN — продолжаем спринты 2-5 (loss hygiene + sample weights + capacity + anchored)" if all_pass else "✗ ESCAPE TO DATA: проблема не в архитектуре, а в признаках/задаче. Архитектурные фиксы не помогут."}')

if not S13_PASS:
    print('\n→ Действие: дебажить cost_aware_classification_labels на конкретных строках с большим diff.')
if not S12_PASS:
    print('\n→ Действие: ALGOPACK микроструктурные фичи + калькулярные не несут predictive-сигнала на этих 2 тикерах.')
    print('  Варианты: (1) добавить тикеры (10+), (2) подключить FUTOI/HI2, (3) сменить горизонты, (4) tick-данные.')
if not S11_PASS:
    print('\n→ Действие: реальный val_loss модели не отличим от случайного. Сигнал не отделим от шума.')
    print('  Дополнительно: запустить с n=100 пермутаций и большей train-выборкой — могло не хватить мощности.')